## Project 3

### Instructions
Using any of the three classifiers described in chapter 6 of Natural Language Processing with Python, and any features you can think of, build the best name gender classifier you can. 

Begin by splitting the Names Corpus into three subsets: 
- 500 words for the test set-
- 500 words for the dev-test set
-  the remaining 6900 words for the training set.

Then, starting with the example name gender classifier, make incremental improvements. 

Use the dev-test set to check your progress. 

Once you are satisfied with your classifier, check its final performance on the test set. 
- How does the performance on the test set compare to the performance on the dev-test set?
- Is this what you'd expect?

In [1]:
### importing and downloading the dataset.
import nltk
nltk.download('names')

[nltk_data] Downloading package names to
[nltk_data]     C:\Users\jferrara_personal\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package names is already up-to-date!


True

In [2]:
### importing the rest of what is needed. 
import random
from nltk.corpus import names
from nltk import classify
from nltk.classify import NaiveBayesClassifier
from nltk.classify import DecisionTreeClassifier
from nltk.classify import MaxentClassifier

In [3]:
### Ensuring there is male and female categorization for the names from source.
labeled_names = [(name, 'male') for name in names.words('male.txt')] + \
                [(name, 'female') for name in names.words('female.txt')]

random.shuffle(labeled_names)

In [4]:
#reporducibility
random.seed(42) 
#shifting order to random for pulling testing lists / sub-lists  
random.shuffle(labeled_names)
test_names = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names = labeled_names[1000:]

print("Testing Names: ",len(test_names))
print("Development Test Names: ",len(devtest_names))
print("Trainign Names: " , len(train_names)) ### 44 more words than the project 3 instructions, but should be fine.

Testing Names:  500
Development Test Names:  500
Trainign Names:  6944


### Featrue 1 (From text fook)

In [5]:
### The first Effort at categorizaiotn can come from the last letter of the name. (From book pg. 223)
def gender_features1(name):
    return {'last_letter': name[-1].lower()}

In [6]:
## Using the simple classifer with the last letter for each of the name in each of the pre made lists 
## Apoplying the first gender_features function to the lists for training and comparison.
train_set_last_letter = [(gender_features1(name), gender) for (name, gender) in train_names]
devtest_set_last_letter = [(gender_features1(name), gender) for (name, gender) in devtest_names]
test_set_last_letter = [(gender_features1(name), gender) for (name, gender) in test_names]

In [7]:
#Making a function that uses all three classifyers from the chapter so as develop my features can just run one function to see results
def train_all_classifiers(train_set):
    classifiers = {}
    classifiers['Naive Bayes'] = NaiveBayesClassifier.train(train_set)
    classifiers['Decision Tree'] = DecisionTreeClassifier.train(train_set)
    classifiers['Maximum Entropy'] = MaxentClassifier.train(train_set, algorithm='iis',trace=0,max_iter=10)
    return classifiers


In [8]:
### Training the first featue of just the last letter
classifiers_feat1 = train_all_classifiers(train_set_last_letter)

In [9]:
# Function to evaluate them eash time 
def evaluate_classifiers(classifiers, eval_set):
    results = {}
    for name, clf in classifiers.items():
        results[name] = classify.accuracy(clf, eval_set)
    return results

In [10]:
## looking at results of each for first feature, all ~76% 
results_feat1 = evaluate_classifiers(classifiers_feat1, devtest_set_last_letter)
print(results_feat1)

{'Naive Bayes': 0.75, 'Decision Tree': 0.75, 'Maximum Entropy': 0.75}


In [11]:
### pulling the best of the bunch, in this case they are all the samne score. SO it iwll pull the first. 
best_model_name_feat1 = max(results_feat1, key=results_feat1.get)
best_classifier_feat1 = classifiers_feat1[best_model_name_feat1]

print("Best model:", best_model_name_feat1)

Best model: Naive Bayes


In [12]:
## Taking a look at the errors in the categorization to work on the feature 
# Collect misclassified examples from the raw dev-test names
def get_errors(classifier, feature_function, devtest_names):
    errors = []
    for (name, actual_gender) in devtest_names:
        predicted_gender = classifier.classify(feature_function(name))
        if predicted_gender != actual_gender:
            errors.append((name, actual_gender, predicted_gender))
    return errors

In [13]:
### printing the actual errors of cetegorizeion with first feature
errors_feat1 = get_errors(best_classifier_feat1, gender_features1, devtest_names)
len(errors_feat1), errors_feat1[:30]

(125,
 [('Jillian', 'female', 'male'),
  ('Derby', 'male', 'female'),
  ('Carroll', 'female', 'male'),
  ('Wylie', 'male', 'female'),
  ('Nan', 'female', 'male'),
  ('Marit', 'female', 'male'),
  ('Sansone', 'male', 'female'),
  ('Ninon', 'female', 'male'),
  ('Jorge', 'male', 'female'),
  ('Ros', 'female', 'male'),
  ('Bridgett', 'female', 'male'),
  ('Jake', 'male', 'female'),
  ('Abe', 'male', 'female'),
  ('Iris', 'female', 'male'),
  ('Tally', 'male', 'female'),
  ('Mab', 'female', 'male'),
  ('Margot', 'female', 'male'),
  ('Towney', 'male', 'female'),
  ('Chad', 'female', 'male'),
  ('Neddy', 'male', 'female'),
  ('Ronnie', 'male', 'female'),
  ('Juanita', 'male', 'female'),
  ('Sawyere', 'male', 'female'),
  ('Billy', 'male', 'female'),
  ('Shanon', 'female', 'male'),
  ('Lyndel', 'female', 'male'),
  ('Jonah', 'male', 'female'),
  ('Adelind', 'female', 'male'),
  ('Harriett', 'female', 'male'),
  ('Nikolai', 'male', 'female')])

##### First Attempt, overview. This modeling attempt only uses the last letter of the names in the training set in order to try and predict the gender of the name inline with the textbook.

##### It does pretty well, with each of the modeling classifiers listed in chapter6 get about ~75 percent accuracy. Beyond this feature, were going to try different features that well engineer to see if we can improve the prediction abilities. We'll consider this a baseline.

### Second Attempt at Feature Extraction

In [14]:
## Desiging the second feature for better cetergoization based on this informaiton above. 
def gender_features2(name):
    name = name.lower()
    return {
        ### Capturing the first letter as well, so that it can be considered with the last one like in feature 1
        'first_letter': name[0],
        ### Same feature for feature 1, but with the additions above and below
        # 'last_letter': name[-1], ## LEAVING THIS ONE OUT TO TRY AND GET A GOOD READS ON THE OTHER FEATURES
        ### Expanding to last 2 letters for portential gendered suffixes
        'last_two': name[-2:] if len(name) >= 2 else name,
        ### looking at the lenght of the name, this may influence the gender ofname 
        'name_length': len(name),
        ### Vowel endigns or not. 
        'ends_with_vowel': name[-1] in 'aeiou'
    }

In [15]:
## Subset definition
train_set_2 = [(gender_features2(name), gender) for (name, gender) in train_names]
devtest_set_2 = [(gender_features2(name), gender) for (name, gender) in devtest_names]
test_set_2 = [(gender_features2(name), gender) for (name, gender) in test_names]

In [16]:
# classifying
classifiers_feat2 = train_all_classifiers(train_set_2)

In [17]:
#evaluate accuracy scores on devtest
results_feat2 = evaluate_classifiers(classifiers_feat2, devtest_set_2)
print(results_feat2)

{'Naive Bayes': 0.746, 'Decision Tree': 0.76, 'Maximum Entropy': 0.792}


In [18]:
### pulling the best of the bunch, in this case they are all the samne score. SO it iwll pull the first. 
best_model_name_feat2 = max(results_feat2, key=results_feat2.get)
best_classifier_feat2 = classifiers_feat2[best_model_name_feat2]

print("Best model:", best_model_name_feat2)

Best model: Maximum Entropy


In [19]:
## informative features 
best_classifier_feat2.show_most_informative_features(100)

  -5.659 last_two=='na' and label is 'male'
  -5.608 last_two=='la' and label is 'male'
  -4.904 last_two=='ia' and label is 'male'
  -4.599 last_two=='ta' and label is 'male'
  -4.379 last_two=='sa' and label is 'male'
  -4.371 last_two=='do' and label is 'female'
  -3.761 last_two=='ra' and label is 'male'
  -3.561 last_two=='io' and label is 'female'
  -3.286 last_two=='ea' and label is 'male'
  -3.213 last_two=='us' and label is 'female'
  -3.192 last_two=='ka' and label is 'male'
  -3.172 last_two=='rd' and label is 'female'
  -2.895 last_two=='ld' and label is 'female'
  -2.820 last_two=='ya' and label is 'male'
  -2.777 last_two=='ti' and label is 'male'
  -2.682 last_two=='rt' and label is 'female'
   2.433 last_two=='eu' and label is 'male'
  -2.361 last_two=='yn' and label is 'male'
  -2.344 last_two=='ns' and label is 'female'
   2.329 last_two=='lu' and label is 'male'
   2.327 last_two=='zo' and label is 'male'
  -2.293 last_two=='ud' and label is 'female'
  -2.273 last_tw

In [20]:
### printing the actual errors of cetegorizeion
errors_feat2 = get_errors(best_classifier_feat2, gender_features2, devtest_names)
len(errors_feat2), errors_feat2[:30]

(104,
 [('Jillian', 'female', 'male'),
  ('Derby', 'male', 'female'),
  ('Nan', 'female', 'male'),
  ('Sansone', 'male', 'female'),
  ('Grayce', 'female', 'male'),
  ('Nancey', 'female', 'male'),
  ('Ninon', 'female', 'male'),
  ('Jorge', 'male', 'female'),
  ('Ros', 'female', 'male'),
  ('Rosemary', 'female', 'male'),
  ('Kevin', 'male', 'female'),
  ('Bridgett', 'female', 'male'),
  ('Abe', 'male', 'female'),
  ('Tally', 'male', 'female'),
  ('Margot', 'female', 'male'),
  ('Wyn', 'male', 'female'),
  ('Chad', 'female', 'male'),
  ('Neddy', 'male', 'female'),
  ('Ronnie', 'male', 'female'),
  ('Juanita', 'male', 'female'),
  ('Sawyere', 'male', 'female'),
  ('Billy', 'male', 'female'),
  ('Shanon', 'female', 'male'),
  ('Jonah', 'male', 'female'),
  ('Adelind', 'female', 'male'),
  ('Harriett', 'female', 'male'),
  ('Rey', 'female', 'male'),
  ('May', 'female', 'male'),
  ('Daffy', 'female', 'male'),
  ('Gail', 'female', 'male')])

##### Now after removing the "last letter" feature, but using the last 2 letters, along with other features like the length of the name, if the name ends with a vowel, and the first letter of the name. The classifiers all get from 74.6% through ~79% accuracy. This is an improvement from the first. The last two letters of the name were the most informative indicators. Lets see how it does if we expand that to the last three. Additionally, were going to try expanding the first letter to first two, and a few other shifts in the features to see how it performs.

### Third Attempt at Feature Extraction

In [70]:
def gender_features3(name):
    name = name.lower()
    return {
        # Expanding to the last three leters of the name where the name has atleast three letters
        'last_three': name[-3:] if len(name) >= 3 else name,
        ## does the name end wit hy
        'ends_with_y': name[-1] == 'y',
        ## does the name end with a vowel (not y)
        'ends_with_vowel': name[-1] in 'aeiou',
        ## does the name have a double letter in it
        'has_double_letter': any(name[i] == name[i+1] for i in range(len(name)-1)),
        ## the first two letters of the name (prefix-oriented)
        'first_two':name[:2]
    }

In [71]:
## Sub lists for testing
train_set_3 = [(gender_features3(name), gender) for (name, gender) in train_names]
devtest_set_3 = [(gender_features3(name), gender) for (name, gender) in devtest_names]
test_set_3 = [(gender_features3(name), gender) for (name, gender) in test_names]

In [72]:
classifiers_feat3 = train_all_classifiers(train_set_3)

In [73]:
## #evaluate accuracy scores on devtest

results_feat3 = evaluate_classifiers(classifiers_feat3, devtest_set_3)
print(results_feat3)

{'Naive Bayes': 0.77, 'Decision Tree': 0.706, 'Maximum Entropy': 0.768}


In [25]:
best_model_name_feat3 = max(results_feat3, key=results_feat3.get)
best_classifier_feat3 = classifiers_feat3[best_model_name_feat3]

print("Best model:", best_model_name_feat3)

Best model: Naive Bayes


In [26]:
best_classifier_feat3.show_most_informative_features(200)

Most Informative Features
              last_three = 'ita'          female : male   =     26.4 : 1.0
              last_three = 'ana'          female : male   =     26.0 : 1.0
              last_three = 'tta'          female : male   =     25.5 : 1.0
              last_three = 'ard'            male : female =     24.7 : 1.0
              last_three = 'nne'          female : male   =     19.0 : 1.0
               first_two = 'hu'             male : female =     15.8 : 1.0
              last_three = 'old'            male : female =     15.8 : 1.0
              last_three = 'son'            male : female =     14.2 : 1.0
              last_three = 'ela'          female : male   =     14.2 : 1.0
               first_two = 'ya'             male : female =     12.5 : 1.0
              last_three = 'dra'          female : male   =     12.4 : 1.0
              last_three = 'ria'          female : male   =     12.4 : 1.0
              last_three = 'vin'            male : female =     10.7 : 1.0

In [27]:
### printing the actual errors of cetegorizeion
errors_feat3 = get_errors(best_classifier_feat3, gender_features3, devtest_names)
len(errors_feat3), errors_feat3[:30]

(109,
 [('Jillian', 'female', 'male'),
  ('Derby', 'male', 'female'),
  ('Wylie', 'male', 'female'),
  ('Nan', 'female', 'male'),
  ('Romeo', 'male', 'female'),
  ('Sansone', 'male', 'female'),
  ('Zea', 'female', 'male'),
  ('Ninon', 'female', 'male'),
  ('Jorge', 'male', 'female'),
  ('Ros', 'female', 'male'),
  ('Bridgett', 'female', 'male'),
  ('Jake', 'male', 'female'),
  ('Iris', 'female', 'male'),
  ('Rici', 'female', 'male'),
  ('Tally', 'male', 'female'),
  ('Mab', 'female', 'male'),
  ('Margot', 'female', 'male'),
  ('Dale', 'female', 'male'),
  ('Towney', 'male', 'female'),
  ('Chad', 'female', 'male'),
  ('Neddy', 'male', 'female'),
  ('Ronnie', 'male', 'female'),
  ('Juanita', 'male', 'female'),
  ('Sawyere', 'male', 'female'),
  ('Billy', 'male', 'female'),
  ('Shanon', 'female', 'male'),
  ('Jonah', 'male', 'female'),
  ('Adelind', 'female', 'male'),
  ('Harriett', 'female', 'male'),
  ('Rey', 'female', 'male')])

##### This attempt added longer suffix information and several structural Boolean features, including whether the name ends with `y`, whether it ends with a vowel (excluding y), whether it contains a double letter, and the first two letters of the name (prefix focused).

##### There is a slight improvement in the Naive Bayes model and the Entropy classifiers. The Decision tree actually decreased, as did the Maximum Entropy classifier score. THe Naive Bayes got better. The Naive Bayes having a 77% accuracy. The maxium entropy also scored around the same as the Naive Bayes. Moving onto the fourth round we're going to try even more features that havent been attempted yet.

##### This feature set did not outperform the best model from the second attempt. While some of the new features appeared informative, the overall accuracy was lower than the best result from feature set 2.

### Fourth Round of Modeling

In [74]:
# Custom Features for this round
def has_consecutive_vowels(name):
    vowels = "aeiou"
    name = name.lower()
    
    for i in range(len(name) - 1):
        if name[i] in vowels and name[i+1] in vowels:
            return True
    return False

def has_double_consonant(name):
    vowels = "aeiou"
    name = name.lower()
    
    for i in range(len(name) - 1):
        if (
            name[i] == name[i+1] and   # same letter
            name[i].isalpha() and
            name[i] not in vowels      # consonant
        ):
            return True
    return False

def has_double_vowel(name):
    vowels = "aeiou"
    name = name.lower()
    
    for i in range(len(name) - 1):
        if (
            name[i] == name[i+1] and   # same letter
            name[i] in vowels          # vowel
        ):
            return True
    return False

def is_vowel_heavy(name):
    vowels = "aeiou"
    name = name.lower()
    
    # Only apply to longer names
    if len(name) < 4:
        return False
    
    vowel_count = sum(1 for ch in name if ch in vowels)
    total_letters = sum(1 for ch in name if ch.isalpha())
    
    if total_letters == 0:
        return False
    
    return (vowel_count / total_letters) >= 0.45

def starts_and_ends_with_vowel(name):
    vowels = "aeiouy"
    name = name.lower()
    
    if len(name) == 0:
        return False
    
    return (name[0] in vowels) and (name[-1] in vowels)

In [29]:
def gender_features4(name):
    name = name.lower()
    return {
        ## Does trhe name have consecutive vowels in it. Doesnt have to be the same.
        'has_consecutive_vowels': has_consecutive_vowels(name),
        ## Does the name have double consonants in it, particularly the same letter
        'has_double_consonant': has_double_consonant(name),
        ## Does the namne have double vowel in it, particularly the same letter
        'has_double_vowel': has_double_vowel(name),
        ## is the word vowel heavy, more then 45% vowels
        'is_vowel_heavy':is_vowel_heavy(name),
        ## does the name start and end with a vowel.
        'starts_and_ends_with_vowel':starts_and_ends_with_vowel(name)
    }

In [30]:
## Sub lists
train_set_4 = [(gender_features4(name), gender) for (name, gender) in train_names]
devtest_set_4 = [(gender_features4(name), gender) for (name, gender) in devtest_names]
test_set_4 = [(gender_features4(name), gender) for (name, gender) in test_names]

In [31]:
### Training
classifiers_feat4 = train_all_classifiers(train_set_4)

In [32]:
## looking at results
results_feat4 = evaluate_classifiers(classifiers_feat4, devtest_set_4)
print(results_feat4)

{'Naive Bayes': 0.646, 'Decision Tree': 0.67, 'Maximum Entropy': 0.67}


In [33]:
best_model_name_feat4 = max(results_feat4, key=results_feat4.get)
best_classifier_feat4 = classifiers_feat4[best_model_name_feat4]

print("Best model:", best_model_name_feat4) ### DT and ME were the same score,

Best model: Decision Tree


In [36]:
errors_feat4 = get_errors(best_classifier_feat4, gender_features4, devtest_names)
len(errors_feat4), errors_feat4[:30]

(165,
 [('Derby', 'male', 'female'),
  ('Wylie', 'male', 'female'),
  ('Romeo', 'male', 'female'),
  ('Sansone', 'male', 'female'),
  ('Lucius', 'male', 'female'),
  ('Francisco', 'male', 'female'),
  ('Ragnar', 'male', 'female'),
  ('Jorge', 'male', 'female'),
  ('Emerson', 'male', 'female'),
  ('Jerrold', 'male', 'female'),
  ('Kevin', 'male', 'female'),
  ('Jake', 'male', 'female'),
  ('Erhard', 'male', 'female'),
  ('Slim', 'male', 'female'),
  ('Abe', 'male', 'female'),
  ('Albrecht', 'male', 'female'),
  ('Tally', 'male', 'female'),
  ('Berk', 'male', 'female'),
  ('Wyn', 'male', 'female'),
  ('Towney', 'male', 'female'),
  ('Theodoric', 'male', 'female'),
  ('Neddy', 'male', 'female'),
  ('Ronnie', 'male', 'female'),
  ('Juanita', 'male', 'female'),
  ('Fabian', 'male', 'female'),
  ('Sawyere', 'male', 'female'),
  ('Billy', 'male', 'female'),
  ('Arnold', 'male', 'female'),
  ('Philip', 'male', 'female'),
  ('Baxter', 'male', 'female')])

In [44]:
## Top model was the DT & ME, but DT is first in dict. switching to see most informative features.
classifiers_feat4['Maximum Entropy'].show_most_informative_features(200)

  -0.775 starts_and_ends_with_vowel==True and label is 'male'
  -0.638 is_vowel_heavy==True and label is 'male'
  -0.557 has_double_consonant==True and label is 'male'
  -0.318 has_double_vowel==True and label is 'male'
   0.284 starts_and_ends_with_vowel==True and label is 'female'
   0.276 is_vowel_heavy==True and label is 'female'
   0.273 has_double_consonant==True and label is 'female'
   0.166 has_double_vowel==True and label is 'female'
   0.122 is_vowel_heavy==False and label is 'male'
  -0.089 is_vowel_heavy==False and label is 'female'
  -0.083 has_consecutive_vowels==False and label is 'male'
  -0.072 has_double_vowel==False and label is 'male'
   0.065 has_consecutive_vowels==False and label is 'female'
   0.058 has_double_vowel==False and label is 'female'
  -0.056 has_consecutive_vowels==True and label is 'male'
   0.056 has_consecutive_vowels==True and label is 'female'
   0.040 has_double_consonant==False and label is 'male'
   0.028 starts_and_ends_with_vowel==False an

##### This attempt focused on broader internal spelling patterns, such as repeated vowels, repeated consonants, vowel-heaviness, and whether a name starts and ends with a vowel. 

##### The attempt did notably worse than models previous. There is no truely clear answer for the most informative features either, the suffix and prefix features seem to do better. None of these will be carried forward to the next modeling atempt


### Fifth Categorization Attempt

In [76]:
def gender_features5(name):
    name = name.lower()
    return {
        'last_three': name[-3:] if len(name) >= 3 else name,
        'first_three':name[:3],
        'ends_with_vowel': name[-1] in 'aeiou',
        'ends_with_y': name[-1] == 'y',
    }

In [77]:
## Subsets 
train_set_5 = [(gender_features5(name), gender) for (name, gender) in train_names]
devtest_set_5 = [(gender_features5(name), gender) for (name, gender) in devtest_names]
test_set_5 = [(gender_features5(name), gender) for (name, gender) in test_names]

In [78]:
### Training 
classifiers_feat5 = train_all_classifiers(train_set_5)

In [79]:
## looking at results
results_feat5 = evaluate_classifiers(classifiers_feat5, devtest_set_5)
print(results_feat5)

{'Naive Bayes': 0.798, 'Decision Tree': 0.696, 'Maximum Entropy': 0.788}


In [80]:
best_model_name_feat5 = max(results_feat5, key=results_feat5.get)
best_classifier_feat5 = classifiers_feat5[best_model_name_feat5]

print("Best model:", best_model_name_feat5)

Best model: Naive Bayes


In [81]:
## Most inform
best_classifier_feat5.show_most_informative_features(100)

Most Informative Features
              last_three = 'ita'          female : male   =     26.4 : 1.0
              last_three = 'ana'          female : male   =     26.0 : 1.0
              last_three = 'tta'          female : male   =     25.5 : 1.0
              last_three = 'ard'            male : female =     24.7 : 1.0
              last_three = 'nne'          female : male   =     19.0 : 1.0
              last_three = 'old'            male : female =     15.8 : 1.0
             first_three = 'ros'          female : male   =     15.4 : 1.0
              last_three = 'son'            male : female =     14.2 : 1.0
              last_three = 'ela'          female : male   =     14.2 : 1.0
             first_three = 'jan'          female : male   =     13.3 : 1.0
              last_three = 'dra'          female : male   =     12.4 : 1.0
              last_three = 'ria'          female : male   =     12.4 : 1.0
             first_three = 'cat'          female : male   =     11.1 : 1.0

In [82]:
### printing the actual errors of cetegorizeion with first feature
errors_feat5 = get_errors(best_classifier_feat5, gender_features5, devtest_names)
len(errors_feat5), errors_feat5[:30]

(101,
 [('Wylie', 'male', 'female'),
  ('Nan', 'female', 'male'),
  ('Romeo', 'male', 'female'),
  ('Sansone', 'male', 'female'),
  ('Francisco', 'male', 'female'),
  ('Conny', 'female', 'male'),
  ('Jorge', 'male', 'female'),
  ('Ros', 'female', 'male'),
  ('Rici', 'female', 'male'),
  ('Tally', 'male', 'female'),
  ('Dale', 'female', 'male'),
  ('Chad', 'female', 'male'),
  ('Ronnie', 'male', 'female'),
  ('Juanita', 'male', 'female'),
  ('Sawyere', 'male', 'female'),
  ('Kee', 'female', 'male'),
  ('Jonah', 'male', 'female'),
  ('Lainey', 'female', 'male'),
  ('Harriett', 'female', 'male'),
  ('Rey', 'female', 'male'),
  ('May', 'female', 'male'),
  ('Daffy', 'female', 'male'),
  ('Nikolai', 'male', 'female'),
  ('Gail', 'female', 'male'),
  ('Reece', 'male', 'female'),
  ('Hildegaard', 'female', 'male'),
  ('Clemence', 'female', 'male'),
  ('Mead', 'male', 'female'),
  ('Graehme', 'male', 'female'),
  ('Tory', 'female', 'male')])

##### This was the best-performing feature set in the notebook, so I decided to make it the final model. The Naive Bayes classifier achieved approximately 79.8% dev-test accuracy. This is slightly higher than the best result from the second attempt with the Max. Entropy technique (79.2%). The most informative features were dominated by three-letter suffixes, confirming that short suffix patterns are especially important for this task.

### Using the final Classifier Model that performed best for the larger set


In [83]:
final_classifier = best_classifier_feat5
final_test_accuracy = classify.accuracy(final_classifier, test_set_5)

print("Final model:", best_model_name_feat5)
print("Final test accuracy:", final_test_accuracy)

Final model: Naive Bayes
Final test accuracy: 0.824


##### After comparing several feature sets with dev test set, I selected feature set 5 as the final model, is the Naive Bayes classifier has the highest accuracy at 79.8%. 

##### I then evaluated the 5th classification model on the main test set, to see how it perform on a more generalized set. The accuracy on the main test set was even higher at 82.4%. 

##### I did not expect the generalized list to yield a higher classification score than the initial devtest list, i expected it to be a bit lower but around the same level of accuracy. Its possible that the random shuffle done in order to generate the devtest lists selected a subset of names that were harder to categorize than the general list.

##### Overall, suffix-focused features were the most effective for this task, Prefix features did also have an impact too. However, the more complex structural features did not improve performance when used on their own. Infact they made performance worse. This is inline Chapter 6 and the emphasis it places on choosing relevant features and avoiding unnecessary complexity.

In [84]:
## Printing All CLassifiers together
print("Feature Set 1:", results_feat1)
print("Feature Set 2:", results_feat2)
print("Feature Set 3:", results_feat3)
print("Feature Set 4:", results_feat4)
print("Feature Set 5:", results_feat5)

Feature Set 1: {'Naive Bayes': 0.75, 'Decision Tree': 0.75, 'Maximum Entropy': 0.75}
Feature Set 2: {'Naive Bayes': 0.746, 'Decision Tree': 0.76, 'Maximum Entropy': 0.792}
Feature Set 3: {'Naive Bayes': 0.77, 'Decision Tree': 0.706, 'Maximum Entropy': 0.768}
Feature Set 4: {'Naive Bayes': 0.646, 'Decision Tree': 0.67, 'Maximum Entropy': 0.67}
Feature Set 5: {'Naive Bayes': 0.798, 'Decision Tree': 0.696, 'Maximum Entropy': 0.788}


### FInal Break down
##### Dev-test accuracy = 0.798
##### Final test accuracy = 0.824
##### Final test is slightly higher than dev-test, which is a little unusual but still reasonable for a random split of 500-name subsets.